# CREATING TOKENIZER FROM SCRATCH 


# Step 1: Creating Tokens

In [62]:
pip install datasets


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [63]:
pip install -U datasets


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [64]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

In [65]:
raw_text = "\n".join(ds["train"]["text"])

print("Total characters:", len(raw_text))
print(raw_text[:999])

Total characters: 540095682

 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 

 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the gam

#### splitting sentences into different tokens 

In [66]:
import re 
text = "Hello, world! How's everything going today? I'm building a tokenizer."
result = re.split(r'(\s|[.,!?;:(){}\[\]<>\"\'`~@#$%^&*+=/\\|_-])', text)
result = [item.strip() for item in result if item.strip()] # removing wide spaces item.strip gets false for spaces
print(result)

['Hello', ',', 'world', '!', 'How', "'", 's', 'everything', 'going', 'today', '?', 'I', "'", 'm', 'building', 'a', 'tokenizer', '.']


In [67]:
preprocessed = re.split(r'(\s|[.,!?;:(){}\[\]<>\"\'`~@#$%^&*+=/\\|_-])', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['=', 'Valkyria', 'Chronicles', 'III', '=', 'Senjō', 'no', 'Valkyria', '3', ':', 'Unrecorded', 'Chronicles', '(', 'Japanese', ':', '戦場のヴァルキュリア3', ',', 'lit', '.', 'Valkyria', 'of', 'the', 'Battlefield', '3', ')', ',', 'commonly', 'referred', 'to', 'as']
105067749


## Step 2: Creating Token IDs

In [68]:
word = sorted(set(preprocessed))
print(len(word))

608557


In [69]:
vocab = { token:integer for integer, token in enumerate(word) }
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
('#', 2)
('$', 3)
('%', 4)
('&', 5)
("'", 6)
('(', 7)
(')', 8)
('*', 9)
('+', 10)
(',', 11)
('-', 12)
('.', 13)
('/', 14)
('0', 15)
('00', 16)
('000', 17)
('0000', 18)
('00000', 19)
('0000000001', 20)
('000000001975', 21)
('000000001977', 22)
('000000001985', 23)
('000000001986', 24)
('000000001987', 25)
('000000001993', 26)
('000000002000', 27)
('000000002008', 28)
('000000002009', 29)
('000000002010', 30)
('000000065', 31)
('0000001', 32)
('0000005', 33)
('000001', 34)
('000003', 35)
('00001', 36)
('0000102880', 37)
('000014', 38)
('000019', 39)
('00003', 40)
('00004', 41)
('00005', 42)
('00006', 43)
('00007', 44)
('00009', 45)
('0000April', 46)
('0000December', 47)
('0000July', 48)
('0000June', 49)
('0000March', 50)


In [70]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encoder(self, text):
        preprocessed = re.split(
            r'(\s|[.,!?;:(){}\[\]<>\"\'`~@#$%^&*+=/\\|_-])',
            text
        )
        preprocessed = [x.strip() for x in preprocessed if x.strip()]
        return [self.str_to_int[x] for x in preprocessed]

    def decoder(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        return text


In [71]:
tokenizer = SimpleTokenizer(vocab)
text = """""It's the last he painted, you know,"
    Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encoder(text)
print(ids)
tokenizer.decoder(ids)

[1, 1, 184022, 6, 548443, 569574, 499243, 481878, 524333, 11, 588123, 496971, 11, 1, 252942, 13, 149661, 548826, 586275, 525643, 535171, 13]


'" " It \' s the last he painted , you know , " Mrs . Gisburn said with pardonable pride .'

# ADDING SPECIAL CONTEXT TOKENS

In [72]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [73]:
len(vocab.items())
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


('𝕄', 608554)
('𝕡', 608555)
('🖕', 608556)
('<|endoftext|>', 608557)
('<|unk|>', 608558)


In [74]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encoder(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decoder(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [75]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "carrying over a large portion of the work"
text2 = "commonly referred to as Valkyria Chronicles III outside Japan"

text = " <|endoftext|> ".join((text1, text2))

print(text)

carrying over a large portion of the work <|endoftext|> commonly referred to as Valkyria Chronicles III outside Japan


In [76]:
tokenizer.encoder(text)


[437866,
 522862,
 415217,
 499124,
 533141,
 520083,
 569574,
 586694,
 608557,
 444706,
 542532,
 571373,
 423874,
 388042,
 85506,
 176782,
 522733,
 186865]

In [77]:
tokenizer.decoder(tokenizer.encoder(text))

'carrying over a large portion of the work <|endoftext|> commonly referred to as Valkyria Chronicles III outside Japan'